In [ ]:
!pip install face_recognition deepface opencv-python tf-keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.6 MB/s eta 0:00:00
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=273ecad98db2eacf2126a2a6402d6749c6bae386f116ba9eb2d4f454e19ff29d
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-rec

In [2]:
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
from google.colab.patches import cv2_imshow
from google.colab.output import eval_js
from IPython.display import display, Javascript, clear_output
from base64 import b64decode
from deepface import DeepFace

def capture_ui():
    js = Javascript('''
    async function takePhoto(quality) {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const btn = document.createElement('button');

        btn.textContent = '📸 Scan My Face Now';
        btn.style.cssText = 'padding:15px 30px; font-size:18px; background:#e91e63; color:white; border:none; border-radius:10px; cursor:pointer; font-weight:bold; margin-top:10px;';

        video.style.cssText = 'display:block; width:500px; border-radius:10px;';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});

        document.body.appendChild(div);
        div.appendChild(video);
        div.appendChild(btn);
        video.srcObject = stream;
        await video.play();

        return new Promise((resolve) => {
            btn.onclick = () => {
                const canvas = document.createElement('canvas');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                stream.getVideoTracks()[0].stop();
                div.remove();
                resolve(canvas.toDataURL('image/jpeg', quality));
            };
        });
    }
    ''')
    display(js)

def start_monitoring():
    print("AI Monitoring System is Loading... Please wait.")

    while True:
        capture_ui()

        try:
            data = eval_js('takePhoto(0.9)')
            binary = b64decode(data.split(',')[1])
            with open('current.jpg', 'wb') as f:
                f.write(binary)

            clear_output(wait=True)

            frame = cv2.imread('current.jpg')

            results = DeepFace.analyze(frame, actions=['emotion'],
                                       enforce_detection=False,
                                       detector_backend='opencv')

            if not isinstance(results, list):
                results = [results]

            detected = False
            for res in results:
                x, y, w, h = res['region']['x'], res['region']['y'], res['region']['w'], res['region']['h']
                emotion = res['dominant_emotion']

                if emotion in ['happy', 'surprise']:
                    status = "Attentive! 😊"
                    color = (0, 255, 0) # Green
                elif emotion == 'neutral':
                    status = "Normal / Listening 😐"
                    color = (0, 255, 255) # Yellow
                else:
                    status = "Needs Focus / Tired 😴"
                    color = (0, 0, 255) # Red

                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 4)
                cv2.putText(frame, status, (x, y-20), cv2.FONT_HERSHEY_DUPLEX, 0.8, color, 2)
                detected = True

            if detected:
                print(f"Time: {datetime.now().strftime('%H:%M:%S')} - Status Updated ✅")
            else:
                print("⚠️ Face not clear. Please sit in good light.")

            cv2_imshow(frame)
            print("\nClick the button above to Scan Again.")

        except Exception as e:
            print(f"Error: {e}")
            continue

start_monitoring()

AI Monitoring System is Loading... Please wait.


<IPython.core.display.Javascript object>

KeyboardInterrupt: 